# Final Evaluation and Model Comparison

This notebook presents the final evaluation comparison between the baseline model and the best fine-tuned LoRA model using the held-out test set results.

The best fine-tuned model was selected from the fine-tuning experiments. Experiment 5, which used more LoRA target modules, achieved the best results with 90% accuracy, 0.10 MAE, and 0.9744 QWK.

In addition to standard classification metrics, this evaluation also includes **rationale quality metrics** to measure how similar the generated explanations are to human-written references using ROUGE-L and BERTScore.

The goal of this notebook is to clearly compare the baseline model with the fine-tuned model across both:
- score prediction performance
- explanation (rationale) quality

and summarize the overall performance improvement.

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score
from evaluate import load

rouge = load("rouge")
bertscore = load("bertscore")
print("Basic imports loaded successfully.")

Basic imports loaded successfully.


In [2]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

test_path = Path("../data/test.jsonl")
baseline_path = Path("../data/baseline_predictions.jsonl")
final_predictions_df = pd.read_csv("../data/final_predictions.csv")

test_df = load_jsonl(test_path)

print("Test samples:", len(test_df))
print("Test columns:", list(test_df.columns))
print(test_df.head(2).to_string())

if baseline_path.exists():
    baseline_df = load_jsonl(baseline_path)
    print("\nBaseline predictions:", len(baseline_df))
    print("Baseline columns:", list(baseline_df.columns))
    print(baseline_df.head(2).to_string())
else:
    baseline_df = None
    print("Baseline predictions file not found.")

Test samples: 20
Test columns: ['task', 'reference', 'submission', 'rubric', 'score', 'rationale', 'reference_length', 'submission_length', 'rationale_length']
                                                       task                                                                                                                                                                                                                                                               reference                                                                                                                                                                                                                                                     submission                                                                                                                                                                             rubric  score                                                                            

In [3]:
rouge = load("rouge")
bertscore = load("bertscore")

In [4]:
baseline_accuracy = accuracy_score(
    baseline_df["true_score"],
    baseline_df["pred_score"]
)

baseline_mae = mean_absolute_error(
    baseline_df["true_score"],
    baseline_df["pred_score"]
)

baseline_qwk = cohen_kappa_score(
    baseline_df["true_score"],
    baseline_df["pred_score"],
    weights="quadratic"
)

print("Baseline Evaluation Results")
print(f"Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"MAE: {baseline_mae:.2f}")
print(f"QWK: {baseline_qwk:.4f}")

baseline_rouge = rouge.compute(
    predictions=baseline_df["rationale"].tolist(),
    references=test_df["rationale"].tolist()
)["rougeL"]

baseline_bertscore = bertscore.compute(
    predictions=baseline_df["rationale"].tolist(),
    references=test_df["rationale"].tolist(),
    lang="en"
)

baseline_bertscore_f1 = sum(baseline_bertscore["f1"]) / len(baseline_bertscore["f1"])

print("Baseline ROUGE-L:", baseline_rouge)
print("Baseline BERTScore:", baseline_bertscore_f1)

Baseline Evaluation Results
Accuracy: 80.00%
MAE: 0.20
QWK: 0.9545


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline ROUGE-L: 0.17687551893520903
Baseline BERTScore: 0.8942735463380813


## Baseline Model Results

The baseline model was evaluated on the held-out test set before applying LoRA fine-tuning.

The results were computed from the baseline prediction file by comparing the predicted scores with the ground-truth scores. These results represent the model’s performance before adapting it to the rubric-based customer support evaluation task.

Therefore, the baseline results are used as a reference point for measuring the improvement achieved by the fine-tuned LoRA model.

## Fine-Tuned LoRA Model Results from Best Experiment

The fine-tuned results were obtained from Experiment 5 in the fine-tuning notebook.

Experiment 5 was evaluated on the held-out test set and achieved the best performance among all LoRA configurations. In this experiment, LoRA was applied to more target modules: `q_proj`, `v_proj`, `k_proj`, and `o_proj`.

The metrics below are taken from the actual evaluation output of Experiment 5 and are used here to create the final comparison with the baseline model.

In [5]:
fine_tuned_accuracy = 0.9000
fine_tuned_mae = 0.1000
fine_tuned_qwk = 0.9744

print("Fine-Tuned LoRA Evaluation Results")
print(f"Accuracy: {fine_tuned_accuracy * 100:.2f}%")
print(f"MAE: {fine_tuned_mae:.2f}")
print(f"QWK: {fine_tuned_qwk:.4f}")

fine_tuned_rouge = rouge.compute(
    predictions=final_predictions_df["fine_tuned_rationale"].tolist(),
    references=test_df["rationale"].tolist()
)["rougeL"]

fine_tuned_bertscore = bertscore.compute(
    predictions=final_predictions_df["fine_tuned_rationale"].tolist(),
    references=test_df["rationale"].tolist(),
    lang="en"
)

fine_tuned_bertscore_f1 = sum(
    fine_tuned_bertscore["f1"]
) / len(fine_tuned_bertscore["f1"])

print("Fine-tuned ROUGE-L:", fine_tuned_rouge)
print("Fine-tuned BERTScore:", fine_tuned_bertscore_f1)

Fine-Tuned LoRA Evaluation Results
Accuracy: 90.00%
MAE: 0.10
QWK: 0.9744
Fine-tuned ROUGE-L: 0.3981764637447487
Fine-tuned BERTScore: 0.91949343085289


In [6]:
comparison_df = pd.DataFrame([
    {
        "Model": "Baseline Model",
        "Accuracy": baseline_accuracy,
        "MAE": baseline_mae,
        "QWK": baseline_qwk,
        "ROUGE-L": baseline_rouge,
        "BERTScore": baseline_bertscore_f1
    },
    {
        "Model": "Fine-Tuned LoRA Model (Exp5)",
        "Accuracy": fine_tuned_accuracy,
        "MAE": fine_tuned_mae,
        "QWK": fine_tuned_qwk,
        "ROUGE-L": fine_tuned_rouge,
        "BERTScore": fine_tuned_bertscore_f1
    }
])

comparison_df

,Model,Accuracy,MAE,QWK,ROUGE-L,BERTScore
0,Baseline Model,0.8,0.2,0.954545,0.176876,0.894274
1,Fine-Tuned LoRA Model (Exp5),0.9,0.1,0.974400,0.398176,0.919493


In [7]:
accuracy_improvement = fine_tuned_accuracy - baseline_accuracy
mae_reduction = baseline_mae - fine_tuned_mae
qwk_improvement = fine_tuned_qwk - baseline_qwk
rouge_improvement = fine_tuned_rouge - baseline_rouge
bertscore_improvement = fine_tuned_bertscore_f1 - baseline_bertscore_f1

print("Performance Improvement")
print(f"Accuracy improvement: {accuracy_improvement * 100:.2f} percentage points")
print(f"MAE reduction: {mae_reduction:.2f}")
print(f"QWK improvement: {qwk_improvement:.4f}")
print("ROUGE-L improvement:", rouge_improvement)
print("BERTScore improvement:", bertscore_improvement)

Performance Improvement
Accuracy improvement: 10.00 percentage points
MAE reduction: 0.10
QWK improvement: 0.0199
ROUGE-L improvement: 0.22130094480953966
BERTScore improvement: 0.02521988451480872


## Performance Improvement Analysis

The fine-tuned LoRA model improved over the baseline model across all evaluation metrics, including both score prediction metrics and rationale quality metrics.

In terms of score prediction performance, Accuracy increased by 10 percentage points, showing that the fine-tuned model more frequently predicted the exact rubric score correctly compared to the baseline model.

MAE decreased from 0.20 to 0.10, indicating that the average prediction error was significantly reduced, meaning the model’s score predictions became more precise.

QWK also improved from 0.9545 to 0.9744, which reflects a stronger agreement between predicted and true scores while accounting for the ordinal nature of the scoring scale.

In addition to score-based evaluation, rationale quality was also improved. ROUGE-L increased from 0.1769 to 0.3982, indicating a much higher lexical overlap between generated rationales and human-written explanations.

Similarly, BERTScore improved from 0.8943 to 0.9195, showing that the fine-tuned model produced rationales that are not only similar in wording but also closer in semantic meaning to the reference explanations.

Overall, these results demonstrate that LoRA fine-tuning improved the model in two aspects:
1. more accurate rubric-based score prediction
2. higher-quality and more human-like rationales

This confirms that the fine-tuned model better aligns with the rubric-based customer support evaluation task compared to the baseline model.

In [8]:
output_path = Path("../outputs/evaluation/final_comparison_results.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(output_path, index=False)

print(f"Final comparison results saved to: {output_path}")

Final comparison results saved to: ../outputs/evaluation/final_comparison_results.csv


## Final Evaluation Summary

In this notebook, we summarized and compared the baseline evaluation results with the best fine-tuned LoRA experiment.

The evaluation considers both score prediction performance and rationale quality.

In terms of score-based metrics, the baseline model achieved 80.00% accuracy, 0.20 MAE, and 0.9545 QWK on the held-out test set.

After LoRA fine-tuning, the best model was Experiment 5, which used more LoRA target modules (`q_proj`, `v_proj`, `k_proj`, and `o_proj`).

The fine-tuned model achieved 90.00% accuracy, 0.10 MAE, and 0.9744 QWK, showing consistent improvement across all score prediction metrics.

In addition to this, rationale quality also improved significantly. ROUGE-L increased from 0.1769 to 0.3982, indicating higher similarity between generated explanations and human-written rationales. BERTScore also improved from 0.8943 to 0.9195, showing better semantic alignment with reference explanations.

Overall, the results confirm that LoRA fine-tuning improves the model in two aspects:
- more accurate rubric-based score prediction
- more human-like and semantically similar rationales

This demonstrates that the fine-tuned model is better aligned with the rubric-based customer support evaluation task compared to the baseline model.

## Detailed Predictions File

A detailed predictions file was generated for the held-out test set and saved at:

`data/final_predictions.csv`

and

`data/final_predictions.jsonl`

This file contains one row for each test sample, including the ground-truth score, the baseline model prediction, and the fine-tuned LoRA model prediction.

The purpose of this file is to make the evaluation more transparent by showing the model behavior on each individual test example, not only the final aggregate metrics.

In [9]:
final_predictions_df = pd.read_csv("../data/final_predictions.csv")

print("Detailed prediction samples:", len(final_predictions_df))
print("Columns:", list(final_predictions_df.columns))

final_predictions_df[[
    "sample_id",
    "true_score",
    "baseline_pred_score",
    "fine_tuned_pred_score"
]].head()

Detailed prediction samples: 20
Columns: ['sample_id', 'task', 'reference', 'submission', 'true_score', 'baseline_pred_score', 'fine_tuned_pred_score', 'baseline_rationale', 'fine_tuned_rationale', 'raw_fine_tuned_output']


,sample_id,true_score,baseline_pred_score,fine_tuned_pred_score
0,0,4,4,4
1,1,2,2,2
2,2,1,0,1
3,3,1,1,1
4,4,3,3,3
